# Kapitel 2 Ett ML-projekt från början till slut

## Uppgift 1-6
1. De sju stegen översiktligt:
- Formulera problemet och se vad för verktyg som behövs.
- Hitta datakällor och samla in data.
- Analysera datan, hitta mönster, avvikelser och viktiga variabler.
- Förbereda och bearbeta datan, t.ex. göra om text till siffror och hantera saknade värden.
- Träna flera olika enkla modeller och välja de som presterar bäst.
- Finjustera modellen och presentera lösningen.
- Produktionssätta och övervaka systemet.
I verkligheten arbetar man iterativt. Stöter man på problem går man ofta tillbaka till tidigare steg.

2. Vad menas med att en modell produktionssätts?
Att man tar modellen från prototypstadiet och integrerar den i ett levande system där den kan göra förutsägelser på riktig data i drift.

3. Vad är scikit-learn och dess designprinciper?
Scikit-learn är standardbiblioteket för ML i Python.
- Estimator: Lär sig från data via metoden fit().
- Transformer: Förbereder och transformerar data via transform() eller fit_transform().
- Predictor: Gör prediktioner via predict() och utvärderar via score().

4. Vad är TensorFlow och Keras?
TensorFlow är ett bibliotek från Google för beräkningar och djupinlärning. Keras är ett enklare API som ligger ovanpå TensorFlow så man snabbt kan bygga neurala nätverk.

5. Kalle och Stinas dialog:
Stina har rätt. Om man ändrar och anpassar modellen efter testdatan försvinner testdatans syfte, och modellen överanpassas till testdatan istället för att generalisera till verklig ny data.

6. Varför många ML-projekt inte når målen:
Ofta beror det på dålig datakvalitet, för lite data, otydliga mål eller att problemet inte lämpade sig för maskininlärning. Man bör börja enkelt med en snabb prototyp för att se om idén fungerar i praktiken.

## Uppgift 8
Koden skapar ett syntetiskt dataset, tränar en linjär regression, sparar den tränade modellen till en fil med joblib och laddar sedan in den igen för att göra prediktioner.
Det är viktigt att kunna spara en modell så man slipper träna om den varje gång den ska användas i produktion eller i en applikation.

In [1]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1, random_state=42)

model = LinearRegression().fit(X, y)
dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")

print(model_loaded.predict(X[:5]))

[ 105.19825923 -124.46546207  -15.07418334  103.66450947   92.8938913 ]


## Uppgift 9
Steg-för-steg modellering av datasetet data_01.csv.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# a) Läs in datasetet
df = pd.read_csv("data_01.csv")

# b) Dela upp i X och y
X = df.drop(columns='target')
y = df['target']

# c) Dela upp i träning, validering och test (20% test, 15% validering av resten)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

# d) Träna två regressionsmodeller
m1 = LinearRegression().fit(X_train, y_train)
m2 = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

# e) Utvärdera modellerna på valideringsdatan
error_m1 = root_mean_squared_error(y_val, m1.predict(X_val))
error_m2 = root_mean_squared_error(y_val, m2.predict(X_val))
print(f"Valideringsfel Linjär regression (RMSE): {error_m1:.2f}")
print(f"Valideringsfel Beslutsträd (RMSE): {error_m2:.2f}")

# f) Träna om den bäst presterande modellen (Linjär regression) på träning + validering
m1.fit(X_train_full, y_train_full)

# g) Utvärdera på testdatan
error_test = root_mean_squared_error(y_test, m1.predict(X_test))
print(f"Testfel Linjär regression (RMSE): {error_test:.2f}")

# h) Träna om modellen på hela datasetet
m1.fit(X, y)

Valideringsfel Linjär regression (RMSE): 3.59
Valideringsfel Beslutsträd (RMSE): 99.36
Testfel Linjär regression (RMSE): 3.37


## Uppgift 10
Modellering och korsvalidering av salary_dataset.csv.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# a) Läs in datasetet och välj X och y
df = pd.read_csv("salary_dataset.csv")
X = df[['YearsExperience']]
y = df['Salary']

# b) Dela upp i träning och test (inget validerings-set)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# c) Träna två modeller med korsvalidering (cv=5)
results_reg = cross_validate(LinearRegression(), X_train, y_train, scoring="neg_root_mean_squared_error", cv=5)
results_tree = cross_validate(DecisionTreeRegressor(max_depth=3, random_state=42), X_train, y_train, scoring="neg_root_mean_squared_error", cv=5)

rmse_reg = -results_reg["test_score"].mean()
rmse_tree = -results_tree["test_score"].mean()
print(f"Linear Regression CV RMSE: {rmse_reg:.2f}")
print(f"Decision Tree CV RMSE: {rmse_tree:.2f}")

# d) Utvärdera den bäst presterande modellen på testsetet
best_model = LinearRegression()
best_model.fit(X_train, y_train)
y_pred_test = best_model.predict(X_test)
test_rmse = root_mean_squared_error(y_test, y_pred_test)
print(f"Test RMSE: {test_rmse:.2f}")

Linear Regression CV RMSE: 5293.20
Decision Tree CV RMSE: 6193.40
Test RMSE: 7059.04


## Uppgift 11
Hantering av kategorisk data med mpg-datasetet.

In [4]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# a) Läs in datasetet mpg
df = sns.load_dataset("mpg")

# b) Droppa saknade värden
df = df.dropna()

# c) Droppa kolumnen name
df = df.drop(columns=['name'])

# d) Dummy-variable-encoding på origin med drop_first=True
df = pd.get_dummies(df, columns=['origin'], drop_first=True)

# e) Dela upp i X och y (mpg)
X = df.drop(columns=['mpg'])
y = df['mpg']

# f) Dela upp i träning och test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# g) Träna linjär regression och utvärdera på testdatan
model = LinearRegression()
model.fit(X_train, y_train)
test_rmse = root_mean_squared_error(y_test, model.predict(X_test))
print(f"Test RMSE: {test_rmse:.2f}")

Test RMSE: 3.26


## Uppgift 12
Förbättra modellen på housing.csv med GridSearchCV.

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor

housing = pd.read_csv('../data/housing.csv').dropna()
X = pd.get_dummies(housing.drop(columns='median_house_value'), drop_first=True)
y = housing['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10]
}

grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='neg_root_mean_squared_error')
grid.fit(X_train, y_train)

print(f"Bästa parametrar: {grid.best_params_}")
print(f"Bästa CV RMSE: {-grid.best_score_:.2f}")

Bästa parametrar: {'max_depth': None, 'n_estimators': 100}
Bästa CV RMSE: 49990.87
